# autofollowdown — CPU-only demo (real results, no GPU)

Everything here runs on a **plain CPU** in ~1–2 minutes — no GPU, no Colab needed. Every number below is measured from a real model.

It shows the full flow on the always-available **native** engine: the three compression techniques → the one-command benchmark → the auto-picker → a real small-LLM perplexity check.

(The specialized GPU backends — NNI / llm-compressor / NVIDIA ModelOpt — are shown in the [Colab T4 notebook](autofollowdown_backends_colab.ipynb).)

Repo: https://github.com/peetwan/autofollowdown

## Setup

In [1]:
# On a fresh machine / Colab, install first (uncomment):
# !pip install "autofollowdown[examples]"
try:
    import autofollowdown as afd
except ImportError:
    import os, sys; sys.path.insert(0, os.path.abspath('..')); import autofollowdown as afd
import warnings; warnings.filterwarnings('ignore')
try:
    import transformers; transformers.logging.set_verbosity_error()
except Exception:
    pass
import torch
print('autofollowdown', afd.__version__, '| CUDA available:', torch.cuda.is_available(), '(CPU-only demo)')

/Users/peet/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


autofollowdown 0.1.0 | CUDA available: False (CPU-only demo)


## 1. The three techniques on a real CNN (trained on CPU)

We train one small CNN on the scikit-learn `digits` dataset (offline) and apply each technique, measuring the real effect on accuracy, size, and sparsity.

In [2]:
import copy, torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from autofollowdown import (ModelCompressor, count_parameters,
                            evaluate_accuracy, model_disk_size_mb)

torch.manual_seed(0)
d = load_digits()
X = torch.tensor(d.images.astype('float32')/16.0).unsqueeze(1)
y = torch.tensor(d.target, dtype=torch.long)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(Xte, yte), batch_size=64)

class CNN(nn.Module):
    def __init__(self, w=32):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1,w,3,padding=1), nn.ReLU(),
                               nn.Conv2d(w,w,3,padding=1), nn.ReLU())
        self.c = nn.Sequential(nn.Flatten(), nn.Linear(w*8*8,128), nn.ReLU(), nn.Linear(128,10))
    def forward(self, x): return self.c(self.f(x))

def train(m, epochs=6):
    opt = torch.optim.Adam(m.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(); m.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            opt.zero_grad(); lossf(m(xb), yb).backward(); opt.step()
    return m.eval()

baseline = train(CNN())
base_acc, base_size = evaluate_accuracy(baseline, test_loader), model_disk_size_mb(baseline)
print(f'baseline: accuracy {base_acc:.1%}, size {base_size:.3f} MB')

baseline: accuracy 95.6%, size 1.048 MB


**Quantization (INT8)** — smaller, accuracy preserved:

In [3]:
q = ModelCompressor(copy.deepcopy(baseline)).quantize(method='int8', approach='dynamic').model
print(f'size {base_size:.3f} MB -> {model_disk_size_mb(q):.3f} MB '
      f'({base_size/model_disk_size_mb(q):.2f}x smaller) | accuracy {base_acc:.1%} -> '
      f'{evaluate_accuracy(q, test_loader):.1%}')

size 1.048 MB -> 0.295 MB (3.55x smaller) | accuracy 95.6% -> 95.6%


[W606 23:33:24.052694000 qlinear_dynamic.cpp:252] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


**Pruning (50% unstructured)** — half the weights become exactly zero:

In [4]:
p = copy.deepcopy(baseline)
_, _, s0 = count_parameters(p)
ModelCompressor(p).prune(sparsity=0.5, method='unstructured')
_, _, s1 = count_parameters(p)
print(f'sparsity {s0:.0%} -> {s1:.0%} | accuracy {base_acc:.1%} -> {evaluate_accuracy(p, test_loader):.1%}')

sparsity 0% -> 50% | accuracy 95.6% -> 94.4%


**Distillation** — a 1/4-width student learns from the teacher:

In [5]:
student = CNN(w=8)
ModelCompressor(student).distill(teacher_model=baseline, train_loader=train_loader, epochs=6, alpha=0.7)
print(f'teacher {base_acc:.1%} / {base_size:.3f} MB  ->  student '
      f'{evaluate_accuracy(student, test_loader):.1%} / {model_disk_size_mb(student):.3f} MB '
      f'({base_size/model_disk_size_mb(student):.2f}x smaller)')

teacher 95.6% / 1.048 MB  ->  student 90.4% / 0.264 MB (3.97x smaller)


## 2. One command: compress every way, benchmark, pick

`compress_and_benchmark` applies all methods, benchmarks them on CPU, and recommends the best size/quality trade-off.

In [6]:
from autofollowdown import compress_and_benchmark
from IPython.display import Markdown, display
study = compress_and_benchmark(baseline, eval_loader=test_loader)
display(Markdown(study.to_markdown()))
print('Recommended:', study.recommended)

| Model | Size (MB) | Params | Sparsity | Latency (ms) | Acc | Fidelity | Size× | Speed× | ΔAcc |
|---|---|---|---|---|---|---|---|---|---|
| baseline | 1.048 | 273,130 | 0.0% | 0.03 | 95.56% | 100.00% | — | — | — |
| int8 dynamic | 0.295 | 9,568 | 0.0% | 0.38 | 95.56% | 99.56% | 3.55× | 0.08× | +0.00% |
| prune 50% | 1.048 | 273,130 | 50.0% | 0.03 | 94.44% | 98.22% | 1.00× | 0.99× | -1.11% |
| prune+quantize | 0.295 | 9,568 | 17.8% | 0.38 | 94.44% | 98.22% | 3.55× | 0.08× | -1.11% |

Recommended: int8 dynamic


## 3. Auto-picker — which library, and why

On CPU the native engine is the runnable pick; the ranking still shows the ideal library for the model (with how to enable it).

In [7]:
from autofollowdown import explain
print(explain(baseline))

Model profile: ModelProfile(family=cnn, params=273,130, conv=True, transformer=False, hf=False, cuda=False)

Ranked compression backends:
 → 1. [0.60] autofollowdown (native): unstructured-0.5 + int8-dynamic (prune+quantize) — runnable
      Global magnitude pruning then portable INT8 — no GPU needed.
   2. [0.90] Microsoft NNI: L1FilterPruner + ModelSpeedup (structured-prune) — not installed
      Channel/filter pruning that ModelSpeedup turns into a genuinely smaller, faster model (real FLOP reduction).
      install: pip install nni
   3. [0.60] NVIDIA TensorRT Model Optimizer: INT8 PTQ (ptq) — not installed
      Calibrated PTQ (SmoothQuant/AWQ/NVFP4) via mtq.quantize, exportable to TensorRT. Requires an NVIDIA GPU.
      install: pip install nvidia-modelopt

Auto-pick (runnable now): autofollowdown (native)


## 4. A real LLM on CPU — WikiText-2 perplexity

`facebook/opt-125m` is small and `nn.Linear`-based, so INT8 dynamic compresses it and runs on CPU. (Downloads ~250 MB the first time; `pip install datasets` for real WikiText-2.)

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from autofollowdown import evaluate_perplexity, load_wikitext2

MODEL_ID = 'facebook/opt-125m'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).eval()
text = load_wikitext2('test')[:1500]

def measure(m):
    return model_disk_size_mb(m), evaluate_perplexity(m, tok, text, stride=512, max_length=512)
b_size, b_ppl = measure(llm)
qllm = ModelCompressor(copy.deepcopy(llm)).quantize(method='int8', approach='dynamic').model
q_size, q_ppl = measure(qllm)

display(Markdown(
    '| Model | Size (MB) | Perplexity↓ | Size\u00d7 | \u0394PPL |\n'
    '|---|---|---|---|---|\n'
    f'| baseline (fp32) | {b_size:.0f} | {b_ppl:.2f} | \u2014 | \u2014 |\n'
    f'| int8 dynamic | {q_size:.0f} | {q_ppl:.2f} | {b_size/q_size:.2f}\u00d7 | {q_ppl-b_ppl:+.2f} |'
))

| Model | Size (MB) | Perplexity↓ | Size× | ΔPPL |
|---|---|---|---|---|
| baseline (fp32) | 478 | 43.74 | — | — |
| int8 dynamic | 272 | 45.82 | 1.76× | +2.08 |

## Done — all of that ran on CPU

Real quantization, pruning, distillation, a benchmark with a recommended pick, and a real LLM perplexity comparison — no GPU involved. For the GPU-backed library shoot-out (NNI · llm-compressor · NVIDIA ModelOpt), open the Colab T4 notebook.